# Ячейка 1: Импорт и настройка

In [126]:
# Ячейка 1: Импорт и настройка
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score
from itertools import combinations

from config import PERIODS, METHODS, TRADING_DAYS
from utils import cluster_by_cv, equal_weight_metrics, ensure_dirs

# Создаём папки
ensure_dirs(['results/tables', 'results/figures'])

# Ячейка 2: Загрузка отфильтрованных тикеров

In [128]:
# Ячейка 2: Загрузка тикеров
top21_file = 'results/tables/top21_tickers.csv'
if not os.path.exists(top21_file):
    raise FileNotFoundError(f"{top21_file} не найден. Сначала запустите 01_filtering_and_metrics.")
tickers = pd.read_csv(top21_file)['ticker'].tolist()
print(f"Загружено {len(tickers)} акций:", tickers)

Загружено 21 акций: ['SPBE', 'MRKV', 'LENT', 'RBCM', 'OZON', 'STSBP', 'MRKY', 'RTSB', 'MRKU', 'SBER', 'RTSBP', 'TGKN', 'RZSB', 'SBERP', 'ETLN', 'RNFT', 'MTSS', 'SVAV', 'RENI', 'AFLT', 'RTGZ']


# Кластеризация для всех наборов (сохранение CSV и графиков)

In [130]:
# Ячейка 3: Кластеризация для всех 4 наборов
cluster_results = {}      # (period, method) -> DataFrame с колонками ticker, cluster
cv_series_dict = {}       # (period, method) -> Series CV
k_optimal_dict = {}       # (period, method) -> оптимальное k

for period_name in PERIODS.keys():
    for method in METHODS:
        price_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
        if not os.path.exists(price_file):
            print(f"Файл {price_file} не найден. Пропускаем {period_name} {method}")
            continue
        print(f"\nОбработка {period_name} {method}...")
        df_cluster, best_k, cv_series = cluster_by_cv(tickers, price_file, max_k=10)
        out_csv = os.path.join('results/tables', f"clusters_{period_name}_{method}.csv")
        df_cluster.to_csv(out_csv, index=False)
        cluster_results[(period_name, method)] = df_cluster
        cv_series_dict[(period_name, method)] = cv_series
        k_optimal_dict[(period_name, method)] = best_k
        print(f"  Оптимальное K = {best_k}, сохранён {out_csv}")


Обработка 2023_2024 inner...
  Оптимальное K = 4, сохранён results/tables\clusters_2023_2024_inner.csv

Обработка 2023_2024 ffill...
  Оптимальное K = 4, сохранён results/tables\clusters_2023_2024_ffill.csv

Обработка 2024_2025 inner...
  Оптимальное K = 5, сохранён results/tables\clusters_2024_2025_inner.csv

Обработка 2024_2025 ffill...
  Оптимальное K = 5, сохранён results/tables\clusters_2024_2025_ffill.csv


# Сохранение деталей кластеров (с CV) для последующего использования в 05

In [132]:
for (period_name, method), df_cluster in cluster_results.items():
    df_details = df_cluster.copy()
    cv_series = cv_series_dict[(period_name, method)]
    df_details['cv'] = df_details['ticker'].map(cv_series)
    df_details.to_csv(os.path.join('results/tables', f"cluster_details_{period_name}_{method}.csv"), index=False)
    print(f"Сохранены детали кластеров для {period_name} {method}")

Сохранены детали кластеров для 2023_2024 inner
Сохранены детали кластеров для 2023_2024 ffill
Сохранены детали кластеров для 2024_2025 inner
Сохранены детали кластеров для 2024_2025 ffill


# Расчёт ARI (Adjusted Rand Index) между inner и ffill для каждого года

In [134]:
ari_results = {}
for period_name in PERIODS.keys():
    df_inner = cluster_results.get((period_name, 'inner'))
    df_ffill = cluster_results.get((period_name, 'ffill'))
    if df_inner is None or df_ffill is None:
        print(f"{period_name}: недостаточно данных для ARI")
        continue
    merged = df_inner.merge(df_ffill, on='ticker', suffixes=('_inner', '_ffill'))
    ari = adjusted_rand_score(merged['cluster_inner'], merged['cluster_ffill'])
    ari_results[period_name] = ari
    print(f"{period_name}: Adjusted Rand Index = {ari:.4f}")

pd.DataFrame({'period': list(ari_results.keys()), 'ARI': list(ari_results.values())}).to_csv('results/tables/ari_scores.csv', index=False)

2023_2024: Adjusted Rand Index = 1.0000
2024_2025: Adjusted Rand Index = 0.8629


# Формирование равновзвешенных портфелей внутри каждого кластера для каждого набора

In [136]:
# Ячейка 5: Генерация портфелей внутри кластеров
def generate_cluster_portfolios(df_cluster, cv_series, period_name, method):
    """
    Для каждого кластера генерирует все сочетания по 2,3,4 акциям,
    вычисляет равновзвешенные метрики.
    Возвращает словарь: {size: DataFrame с портфелями}
    """
    results = {2: [], 3: [], 4: []}
    for cl in df_cluster['cluster'].unique():
        cluster_tickers = df_cluster[df_cluster['cluster'] == cl]['ticker'].tolist()
        if len(cluster_tickers) < 2:
            continue
        for n in [2,3,4]:
            if len(cluster_tickers) < n:
                continue
            for combo in combinations(cluster_tickers, n):
                # Расчёт равновзвешенных метрик
                price_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
                df_prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
                returns = df_prices.pct_change().dropna()
                mean_ret = returns.mean()
                cov = returns.cov()
                rf = PERIODS[period_name]['rf']
                annual_ret, annual_risk, sharpe = equal_weight_metrics(list(combo), mean_ret, cov, rf)
                record = {f'ticker{i+1}': ticker for i, ticker in enumerate(combo)}
                record['cluster'] = cl
                record['annual_return'] = annual_ret
                record['annual_risk'] = annual_risk
                record['sharpe'] = sharpe
                results[n].append(record)
    # Преобразуем в DataFrame
    for n in [2,3,4]:
        if results[n]:
            results[n] = pd.DataFrame(results[n])
            # Добавим нормализованный ключ
            if n == 2:
                results[n]['pair_key'] = results[n].apply(lambda r: tuple(sorted([r['ticker1'], r['ticker2']])), axis=1)
            elif n == 3:
                results[n]['triple_key'] = results[n].apply(lambda r: tuple(sorted([r['ticker1'], r['ticker2'], r['ticker3']])), axis=1)
            else:
                results[n]['quad_key'] = results[n].apply(lambda r: tuple(sorted([r['ticker1'], r['ticker2'], r['ticker3'], r['ticker4']])), axis=1)
        else:
            results[n] = pd.DataFrame()
    return results

all_portfolios = {}  # (period, method, n) -> DataFrame
for period_name in PERIODS.keys():
    for method in METHODS:
        key = (period_name, method)
        if key not in cluster_results:
            continue
        df_cluster = cluster_results[key]
        cv_series = cv_series_dict[key]
        res = generate_cluster_portfolios(df_cluster, cv_series, period_name, method)
        for n in [2,3,4]:
            if not res[n].empty:
                out_file = os.path.join('results/tables', f"cluster_portfolios_{n}_{period_name}_{method}.csv")
                res[n].to_csv(out_file, index=False)
        all_portfolios[(period_name, method, 2)] = res[2]
        all_portfolios[(period_name, method, 3)] = res[3]
        all_portfolios[(period_name, method, 4)] = res[4]
        print(f"{period_name} {method}: пар={len(res[2])}, троек={len(res[3])}, четвёрок={len(res[4])}")

2023_2024 inner: пар=54, троек=86, четвёрок=90
2023_2024 ffill: пар=54, троек=86, четвёрок=90
2024_2025 inner: пар=35, троек=29, четвёрок=12
2024_2025 ffill: пар=35, троек=29, четвёрок=12


# Поиск общих портфелей для всех четырёх наборов

In [138]:
# Ячейка 6: Поиск общих портфелей
common_portfolios = {2: None, 3: None, 4: None}
for n in [2,3,4]:
    # Собираем множества ключей из каждого набора
    key_sets = []
    for period_name in PERIODS.keys():
        for method in METHODS:
            df = all_portfolios.get((period_name, method, n))
            if df is not None and not df.empty:
                key_col = 'pair_key' if n==2 else ('triple_key' if n==3 else 'quad_key')
                key_sets.append(set(df[key_col]))
    if len(key_sets) == 4:
        common_keys = set.intersection(*key_sets)
        print(f"\nОбщих портфелей размера {n} во всех 4 наборах: {len(common_keys)}")
        if common_keys:
            # Берём данные из первого набора (например, 2023_2024 inner) для демонстрации
            sample_df = all_portfolios.get(('2023_2024', 'inner', n))
            if sample_df is not None and not sample_df.empty:
                key_col = 'pair_key' if n==2 else ('triple_key' if n==3 else 'quad_key')
                common_data = []
                for key in common_keys:
                    row = sample_df[sample_df[key_col] == key].iloc[0]
                    record = {}
                    for i in range(1, n+1):
                        record[f'ticker{i}'] = row[f'ticker{i}']
                    record['cluster'] = row['cluster']  # кластер из первого набора (для информации)
                    record['annual_return'] = row['annual_return']
                    record['annual_risk'] = row['annual_risk']
                    record['sharpe'] = row['sharpe']
                    common_data.append(record)
                df_common = pd.DataFrame(common_data)
                out_file = f'results/tables/common_cluster_portfolios_{n}.csv'
                df_common.to_csv(out_file, index=False)
                print(f"Сохранён {out_file}")
                common_portfolios[n] = df_common
    else:
        print(f"\nНедостаточно данных для поиска общих портфелей размера {n}")


Общих портфелей размера 2 во всех 4 наборах: 6
Сохранён results/tables/common_cluster_portfolios_2.csv

Общих портфелей размера 3 во всех 4 наборах: 1
Сохранён results/tables/common_cluster_portfolios_3.csv

Общих портфелей размера 4 во всех 4 наборах: 0


# Сводка по кластерам (распределение акций) – для диплома

In [140]:
# Ячейка 7: Таблицы распределения акций по кластерам для каждого набора
for (period_name, method), df_cluster in cluster_results.items():
    print(f"\n{period_name} {method}:")
    cluster_counts = df_cluster['cluster'].value_counts().sort_index()
    for cl, cnt in cluster_counts.items():
        tickers_list = df_cluster[df_cluster['cluster'] == cl]['ticker'].tolist()
        # Вычислим средний CV для кластера
        cv_series = cv_series_dict[(period_name, method)]
        mean_cv = cv_series[tickers_list].mean()
        print(f"  Кластер {cl} ({cnt} акций): средний CV = {mean_cv:.4f}")
        print(f"    Тикеры: {tickers_list}")
    # Сохраним подробную таблицу
    df_cluster_with_cv = df_cluster.copy()
    df_cluster_with_cv['cv'] = df_cluster['ticker'].map(cv_series_dict[(period_name, method)])
    df_cluster_with_cv.to_csv(os.path.join('results/tables', f"cluster_details_{period_name}_{method}.csv"), index=False)


2023_2024 inner:
  Кластер 0 (8 акций): средний CV = 8.5558
    Тикеры: ['SPBE', 'MRKV', 'RBCM', 'SBER', 'SBERP', 'RNFT', 'MTSS', 'AFLT']
  Кластер 1 (5 акций): средний CV = 18.5123
    Тикеры: ['MRKY', 'RTSB', 'TGKN', 'SVAV', 'RENI']
  Кластер 2 (2 акций): средний CV = 4.5783
    Тикеры: ['LENT', 'OZON']
  Кластер 3 (6 акций): средний CV = 14.0389
    Тикеры: ['STSBP', 'MRKU', 'RTSBP', 'RZSB', 'ETLN', 'RTGZ']

2023_2024 ffill:
  Кластер 0 (8 акций): средний CV = 8.5952
    Тикеры: ['SPBE', 'MRKV', 'RBCM', 'SBER', 'SBERP', 'RNFT', 'MTSS', 'AFLT']
  Кластер 1 (5 акций): средний CV = 18.5938
    Тикеры: ['MRKY', 'RTSB', 'TGKN', 'SVAV', 'RENI']
  Кластер 2 (2 акций): средний CV = 4.5901
    Тикеры: ['LENT', 'OZON']
  Кластер 3 (6 акций): средний CV = 14.0964
    Тикеры: ['STSBP', 'MRKU', 'RTSBP', 'RZSB', 'ETLN', 'RTGZ']

2024_2025 inner:
  Кластер 0 (4 акций): средний CV = 15.3078
    Тикеры: ['RTSB', 'ETLN', 'RNFT', 'RTGZ']
  Кластер 1 (5 акций): средний CV = 9.9017
    Тикеры: ['MRKV',

# Краткая итоговая сводка

In [142]:
# Ячейка 8: Итог
print("\n=== ИТОГОВАЯ СВОДКА ===")
for n in [2,3,4]:
    if common_portfolios[n] is not None:
        print(f"Общих портфелей размера {n}: {len(common_portfolios[n])}")
    else:
        print(f"Общих портфелей размера {n}: не найдено")


=== ИТОГОВАЯ СВОДКА ===
Общих портфелей размера 2: 6
Общих портфелей размера 3: 1
Общих портфелей размера 4: не найдено
